# 07 - Comparación de las dos formas de elegir el modelo

El profesor nos marcó que lo más difícil del proyecto es **cómo dividir los datos**. Por eso comparamos dos formas de elegir el modelo. Las dos usan los mismos candidatos (las 10 regresiones Ridge del notebook 06 y 3 modelos de árboles con boosting), validan en **los mismos días** (todo 2025), se vuelven a entrenar con 2024 y 2025 y se prueban **una sola vez** en 2026.

| | **Opción 1: holdout de un año** | **Opción 2: validación cruzada temporal** |
|---|---|---|
| Cómo se valida 2025 | Un solo modelo entrenado con 2024 predice todo 2025 | 2025 se parte en 4 trimestres. Cada trimestre lo predice un modelo entrenado con **todo lo anterior** |
| Train de cada parte | Siempre 2024 | 2024 → 2024 + T1 → 2024 + T1 + T2 → 2024 + T1 + T2 + T3 |
| Cómo se elige | Menor MAE en 2025 | Menor MAE en 2025, juntando los 4 trimestres |
| Reentrenamiento final | 2024 + 2025 | 2024 + 2025 |
| Test | 2026, una vez | 2026, una vez |

```
            2024              2025                              2026
            ene ...... dic    T1      T2      T3      T4        ene ..... oct
Opción 1    [ train       ]  [ val ................................ ]
Opción 2    [ train       ]  [ val ]
            [ train ...............] [ val ]
            [ train .......................] [ val ]
            [ train ...............................] [ val ]
Final       [ train ................................................] [ test ]
```

La ventana T1 de la opción 2 tiene el mismo train que la opción 1, así que en enero a marzo las dos tienen que dar exactamente lo mismo. La diferencia aparece de abril en adelante, cuando la opción 2 ya entrena con meses de 2025.

**Qué preguntamos:**
1. ¿Eligen el mismo modelo?
2. El modelo que elige cada una, ¿cuánto se equivoca en 2026?
3. ¿Cuál **anticipa mejor** el error de 2026? Si validación promete un error y en test sale mucho más alto, esa forma de validar es optimista.
4. ¿El orden de los candidatos en validación se mantiene en test?

**Este notebook no toca las tablas de los otros.** Solo lee `demanda_gold` y guarda sus resultados en otra tabla, `comparacion_validacion`. No vuelve a armar gold ni pisa `resultados_modelos`.

**De dónde venimos.** En el 06 elegimos la mejor regresión con train 2024 y val 2025. Ahí vimos que Ridge no sobreajusta y que los días que le cuestan (Turismo, la primera semana de enero, días únicos) piden combinar la línea, la hora y el tipo de día, cosa que una regresión no hace sola. Por eso acá sumamos árboles con boosting. La división del 06 es la opción 1, así que antes de comparar controlamos que las regresiones den lo mismo que allá (sección 3).

Las ventanas están en `config.OPCIONES_VALIDACION` y las funciones en `src/bondis/validacion.py`.

In [0]:
%run ./00_setup

# 00 - Preparar el proyecto

Este notebook deja todo listo para que los demás funcionen. Hace cuatro cosas:

1. Le dice a Python dónde está nuestro código (la carpeta `src/bondis`), para poder escribir `from bondis import ...`.
2. Crea en Databricks el lugar donde se guardan las tablas y los archivos (`bondis_stm.datos.raw`, definido en `config.py`).
3. Crea una carpeta para cada tipo de archivo (viajes, clima, etc.).
4. Muestra cómo dividimos los datos en train, val y test.

Se puede correr todas las veces que quieras. Si algo ya existe, no lo vuelve a crear ni lo borra.

No hace falta abrirlo a mano: los otros notebooks lo corren al principio con `%run ./00_setup`.

## 1. Encontrar nuestro código

Python solo puede importar código de las carpetas que conoce. Esta celda busca la carpeta del repositorio subiendo carpeta por carpeta hasta encontrar `src/bondis`, y la agrega a la lista de carpetas que Python conoce (`sys.path`).

## 2. Crear el catálogo, el esquema y el volume

Databricks ordena todo en tres niveles, como carpetas una dentro de otra:

- **Catálogo** (`bondis_stm`): la carpeta grande del proyecto.
- **Esquema** (`datos`): una subcarpeta donde van las tablas.
- **Volume** (`raw`): una carpeta para guardar archivos sueltos, como los ZIP y CSV que bajamos.


## 3. Crear una carpeta para cada tipo de archivo

| Carpeta | Qué va adentro |
|---|---|
| `viajes_zip/` | Los ZIP de viajes de cada mes, bajados a mano del portal de la Intendencia |
| `viajes/` | Los CSV de viajes, ya sacados de los ZIP |
| `clima/` | Lluvia y temperatura de cada hora |
| `calendario/` | Por ahora vacía. El calendario está escrito en `src/bondis/calendario.py` |
| `paros/` | Vacía. Al final no usamos paros, porque en los de estos años los ómnibus anduvieron normal (ver `calendario.py`) |

## 4. Revisar que todo esté bien

Muestra la versión de Spark, el usuario y cuántos archivos hay en cada carpeta. Sirve para ver de un vistazo si falta subir algo.

## 5. Cómo dividimos los datos

Usamos las fechas para dividir, no el azar, porque queremos predecir el futuro con el pasado. Un año entero para cada cosa:

| Conjunto | Fechas | Para qué se usa |
|---|---|---|
| `train` | todo 2024 | El modelo aprende con estos datos |
| `val` | todo 2025 | Comparamos y elegimos los parámetros de la regresión |
| `test` | 2026 (lo que haya) | Probamos el modelo elegido una sola vez, al final |

La fecha de fin **no** está incluida. Por eso siempre se filtra con `fecha >= inicio` y `fecha < fin`.

`train_final` es train + val juntos. Cuando ya elegimos el modelo, lo volvemos a entrenar con 2024 y 2025 antes de probarlo en 2026.

... (19 outputs left out)

## 2. Crear el catálogo, el esquema y el volume

Databricks ordena todo en tres niveles, como carpetas una dentro de otra:

- **Catálogo** (`bondis_stm`): la carpeta grande del proyecto.
- **Esquema** (`datos`): una subcarpeta donde van las tablas.
- **Volume** (`raw`): una carpeta para guardar archivos sueltos, como los ZIP y CSV que bajamos.


## 3. Crear una carpeta para cada tipo de archivo

| Carpeta | Qué va adentro |
|---|---|
| `viajes_zip/` | Los ZIP de viajes de cada mes, bajados a mano del portal de la Intendencia |
| `viajes/` | Los CSV de viajes, ya sacados de los ZIP |
| `clima/` | Lluvia y temperatura de cada hora |

El calendario (feriados, clases y paros) no tiene carpeta, porque está escrito a mano en `src/bondis/calendario.py`.

## 4. Revisar que todo esté bien

Muestra la versión de Spark y cuántos archivos hay en cada carpeta. Sirve para ver de un vistazo si falta subir algo.

## 5. Cómo dividimos los datos

Usamos las fechas para dividir, no el azar, porque queremos predecir el futuro con el pasado. Un año entero para cada cosa:

| Conjunto | Fechas | Para qué se usa |
|---|---|---|
| `train` | todo 2024 | El modelo aprende con estos datos |
| `val` | todo 2025 | Comparamos y elegimos los parámetros de la regresión |
| `test` | 2026 (lo que haya) | Probamos el modelo elegido una sola vez, al final |

La fecha de fin **no** está incluida. Por eso siempre se filtra con `fecha >= inicio` y `fecha < fin`.

`train_final` es train + val juntos. Cuando ya elegimos el modelo, lo volvemos a entrenar con 2024 y 2025 antes de probarlo en 2026.

In [0]:
import importlib
from datetime import datetime

import matplotlib.pyplot as plt
import pandas as pd
from pyspark.sql import functions as F

from bondis import metricas, modelos, validacion, variables

for _m in (metricas, modelos, validacion, variables):
    importlib.reload(_m)

spark.conf.set("spark.sql.session.timeZone", config.TIMEZONE)
plt.rcParams.update({"figure.figsize": (13, 4), "axes.grid": True, "grid.alpha": 0.3})
pd.set_option("display.width", 200)

REFERENCIA = "mediana_4sem"
REGLAS = ["mediana_4sem", "perfil_historico", "regla_por_tipo_dia"]
# Árboles: poisson, log y residual (la que aprende cuánto corregir la mediana).
# Tardan bastante más que Ridge. Si Databricks no llega, poner False y se corre solo con Ridge.
INCLUIR_ARBOLES = True

CANDIDATOS = validacion.candidatos_ridge()   # los mismos 10 del notebook 06
if INCLUIR_ARBOLES:
    CANDIDATOS.update(validacion.candidatos_arboles())
OPCIONES = config.OPCIONES_VALIDACION

print("Candidatos:", list(CANDIDATOS))
for opcion, ventanas in OPCIONES.items():
    print(f"{opcion}: {[v[0] for v in ventanas]}")

## 1. Datos

Se bajan a pandas las filas completas de 2024 y 2025 (lo que se usa para elegir el modelo). **2026 no se baja todavía**, recién en la sección 5.

El tamaño de cada línea (grande, mediana, chica) se calcula con 2024, igual que en el 06, para que los cortes por tamaño sean los mismos en las dos opciones.

In [0]:
gold = variables.con_tipo_dia(spark.table(config.TABLE_GOLD)).where(F.col("fila_completa"))
primer_dia_val = gold.where(F.col("particion") == "val").agg(F.min("fecha")).first()[0]
assert primer_dia_val is not None and primer_dia_val.year == 2025, \
    "Gold no tiene val en 2025. Hay que volver a correr el notebook 03."

leer, faltantes = modelos.columnas_a_leer(gold.columns)
if faltantes:
    print("Columnas opcionales que gold todavía no tiene (se sigue sin ellas):", faltantes)

tamanos = variables.tamano_linea(gold.where(F.col("particion") == "train")).toPandas()


def bajar(desde, hasta):
    # Filas completas desde 'desde' hasta 'hasta' (sin incluirlo), en pandas, con el tamaño de la línea.
    df = (gold.where((F.col("fecha") >= F.to_date(F.lit(desde))) & (F.col("fecha") < F.to_date(F.lit(hasta))))
          .select(*leer).toPandas()
          .merge(tamanos, on="linea", how="left")
          .fillna({"tamano_linea": "chica"}))
    df["fecha"] = pd.to_datetime(df["fecha"])
    return df


datos = bajar(config.FECHA_INICIO, config.FECHA_INICIO_TEST)
print(f"2024 y 2025: {len(datos):,} filas, {datos['fecha'].nunique()} días "
      f"({datos['fecha'].min().date()} a {datos['fecha'].max().date()}), {datos['linea'].nunique()} líneas")

## 2. Entrenar y predecir 2025 con cada opción

Para cada ventana: se entrenan todos los candidatos con su train, se predice su parte de 2025 y se calculan las reglas simples con las mismas filas. Las líneas y tipos de día que conoce cada modelo salen solo de su train.

La opción 1 entrena cada candidato una vez y la opción 2 cuatro veces (una por trimestre). Las regresiones tardan segundos. Los árboles tardan bastante más (pueden ser uno o dos minutos cada uno), así que con árboles esta celda puede llevar media hora o más.

En la opción 2 cada trimestre entrena con más filas que el anterior, pero el `alfa` de cada regresión queda fijo. Eso importa porque Ridge compara el castigo contra la suma de los errores de todas las filas (ver 06, sección 6.2), así que con más filas el mismo `alfa` castiga menos. Para los `alfa` grandes la opción 2 va a salir mejor solo por eso, no porque los datos de 2025 ayuden.

In [0]:
est_val, detalle = {}, {}
for opcion, ventanas in OPCIONES.items():
    print(f"=== {opcion}")
    est_val[opcion], detalle[opcion] = validacion.evaluar_opcion(datos, ventanas, CANDIDATOS, config.FECHA_INICIO)
    print()

print(pd.concat([d.assign(opcion=o) for o, d in detalle.items()]).to_string(index=False))

**Control.** Las dos opciones tienen que predecir exactamente las mismas filas y días. Además, en el primer trimestre (mismo train) tienen que dar el mismo error.

In [0]:
filas = {o: int(e.loc[e["modelo"] == REFERENCIA, "n"].sum()) for o, e in est_val.items()}
dias = {o: e["fecha"].nunique() for o, e in est_val.items()}
print("Filas de validación:", filas)
print("Días de validación: ", dias)
assert len(set(filas.values())) == 1 and len(set(dias.values())) == 1, "Las opciones no validan las mismas filas"


def error_t1(e):
    t1 = e[validacion.trimestre(e["fecha"]) == "2025_T1"]
    return t1[t1["modelo"].isin(CANDIDATOS)].groupby("modelo")["s_abs"].sum()


dif_t1 = (error_t1(est_val["opcion_1"]) - error_t1(est_val["opcion_2"])).abs().max()
print(f"Diferencia máxima en el primer trimestre: {dif_t1:.6f} (tiene que ser 0)")

## 3. Resultados en validación (2025)

Igual que en el 05 y el 06: el MAE con su intervalo del 95 % (sorteando semanas) y la mejora sobre la mediana de 4 semanas.

La opción 1 tendría que elegir el mismo modelo que el notebook 06, porque es la misma división con las mismas filas.

In [0]:
resumen_val = {o: validacion.resumen(e, REFERENCIA, n_rep=1000, seed=config.SEED) for o, e in est_val.items()}
elegidos = {o: validacion.elegir(r, CANDIDATOS) for o, r in resumen_val.items()}

for o, r in resumen_val.items():
    print(f"=== {o}: elige {elegidos[o]}")
    print(validacion.para_mostrar(r).to_string(index=False))
    print()

if len(set(elegidos.values())) == 1:
    print("Las dos opciones eligen el mismo modelo.")
else:
    print("Las opciones eligen modelos distintos:", elegidos)

**Control contra el 06.** La opción 1 es la misma división que el 06, con las mismas filas y los mismos candidatos de Ridge. Leemos lo que guardó el 06 en `resultados_modelos` y comparamos. Si la diferencia no es casi 0, algo cambió entre las dos corridas (gold, el código o las variables) y hay que volver a correr el 06.

In [0]:
if spark.catalog.tableExists(config.TABLE_RESULTADOS):
    del_06 = (spark.table(config.TABLE_RESULTADOS).where(F.col("etapa") == "modelo")
              .select("modelo", "mae", "seleccionado").toPandas().set_index("modelo"))
    control = del_06[["mae"]].rename(columns={"mae": "mae_06"}).join(
        resumen_val["opcion_1"].set_index("modelo")["mae"].rename("mae_07_opcion_1"), how="inner")
    control["diferencia"] = control["mae_07_opcion_1"] - control["mae_06"]
    print(control.round(3).to_string())
    print(f"\nDiferencia máxima: {control['diferencia'].abs().max():.4f}")
    print("Elegida en el 06:", ", ".join(del_06.index[del_06["seleccionado"]]))
else:
    print("Todavía no está la tabla del 06. Correr el 06 antes para poder comparar.")

### 3.1 Los candidatos lado a lado

El MAE y el puesto de cada candidato en las dos opciones. Si los puestos cambian mucho, la forma de validar cambia la decisión.

In [0]:
lado = pd.concat({o: r.set_index("modelo")[["mae", "mejora_mae"]] for o, r in resumen_val.items()}, axis=1)
lado = lado.loc[[*CANDIDATOS, *REGLAS]]
for o in OPCIONES:
    lado[(o, "puesto")] = lado.loc[list(CANDIDATOS), (o, "mae")].rank(method="min")
lado = lado[[(o, c) for o in OPCIONES for c in ("mae", "mejora_mae", "puesto")]]
print(lado.round(3).to_string())

### 3.1.1 Ridge contra árboles

El mejor de cada tipo de modelo en cada opción. Sirve para ver si la diferencia entre opciones está en elegir entre tipos de modelo o en ajustar los parámetros dentro de un tipo. Abajo de cada tabla está cuánto le gana el mejor árbol a la mejor Ridge, con el mismo bootstrap por semanas. Si el intervalo no incluye el 0, la diferencia no es por los días que tocaron.

In [0]:
def tipo_modelo(nombre):
    return "ridge" if nombre.startswith("lineal") else "árboles"


for o, r in resumen_val.items():
    c = r[r["modelo"].isin(CANDIDATOS)].assign(tipo=lambda t: t["modelo"].map(tipo_modelo))
    mejores = c.loc[c.groupby("tipo")["mae"].idxmin(), ["tipo", "modelo", "mae", "mejora_mae", "mejora_mae_inf", "mejora_mae_sup"]]
    print(f"{o}:")
    print(mejores.round(3).to_string(index=False))
    m = mejores.set_index("tipo")["modelo"]
    if len(m) == 2:   # solo si se corrieron los árboles
        d = validacion.diferencia_entre_opciones(est_val[o], m["árboles"], m["ridge"], n_rep=1000, seed=config.SEED)
        print(f"Mejora de {m['árboles']} sobre {m['ridge']}: {100 * d['mejora']:.1f} % "
              f"[IC 95 %: {100 * d['inf']:.1f} % a {100 * d['sup']:.1f} %]")
    print()

### 3.2 Trimestre por trimestre

La diferencia entre las opciones está en con qué datos se entrena cada trimestre. La tabla muestra, para cada candidato, el MAE de la opción 2 menos el de la opción 1 en cada trimestre. Negativo quiere decir que entrenar con los meses de 2025 ya pasados ayudó.

T1 tiene que dar 0. Lo que esperábamos ver es que la diferencia crezca hacia T4, porque ahí la opción 1 predice con datos de hasta un año atrás.

En el gráfico no van los candidatos que no le ganan a la mediana en ninguna de las dos opciones (los `alfa` más grandes), porque su diferencia es tan grande que aplasta a los demás. En la tabla sí están todos.

In [0]:
por_trim = {}
for o, e in est_val.items():
    m = metricas.metricas(e.assign(trimestre=validacion.trimestre(e["fecha"])), por=["trimestre"])
    por_trim[o] = m.pivot(index="modelo", columns="trimestre", values="mae")

mostrar = list(dict.fromkeys([REFERENCIA, *elegidos.values()]))
for o, t in por_trim.items():
    print(f"MAE por trimestre, {o}:")
    print(t.loc[mostrar].round(2).to_string(), "\n")

dif = (por_trim["opcion_2"] - por_trim["opcion_1"]).loc[list(CANDIDATOS)]
print("MAE opción 2 menos MAE opción 1 (negativo: la opción 2 se equivoca menos):")
print(dif.round(2).to_string())

utiles = [n for n in CANDIDATOS if max(r.set_index("modelo").loc[n, "mejora_mae"] for r in resumen_val.values()) > 0]
fig, ax = plt.subplots(figsize=(8, 4))
for nombre in utiles:
    ax.plot(dif.columns, dif.loc[nombre], marker="o", lw=1, label=nombre)
ax.axhline(0, color="gray", lw=1)
ax.set(title="Diferencia de MAE entre opciones, por trimestre de 2025", ylabel="MAE opción 2 - opción 1")
ax.legend(fontsize=7)
plt.show()

### 3.3 Por tipo de día

Los modelos elegidos y la referencia, separando por tipo de día.

In [0]:
for o, e in est_val.items():
    m = metricas.metricas(e[e["modelo"].isin(mostrar)], por=["tipo_dia"])
    tipos = [t for t in variables.TIPOS_DIA if t in set(m["tipo_dia"])]
    print(f"MAE por tipo de día, {o}:")
    print(m.pivot(index="modelo", columns="tipo_dia", values="mae").loc[mostrar, tipos].round(1).to_string(), "\n")

**Qué vimos en la sección 3.**
- **Las dos opciones eligen el mismo modelo**, `hgb_poisson_h63` (MAE 26,6 en la opción 1 y 26,5 en la opción 2), y los puestos casi no cambian. Solo se dan vuelta `hgb_log_h63` y `hgb_residual_h63`, que están a menos de medio pasajero. Para este problema la forma de validar no cambió la decisión.
- **Los árboles le ganan a todas las regresiones.** El mejor árbol se equivoca un 13,5 % menos que la mejor Ridge en la opción 1 (intervalo de 10,7 % a 16,4 %) y un 16,0 % menos en la opción 2. Como el intervalo no incluye el 0, la diferencia no es por los días que tocaron. Por tipo de día, el árbol mejora a la Ridge del 06 sobre todo en hábiles (29,0 contra 33,3) y en Turismo (28,7 contra 51,9), justo donde la regresión no podía combinar variables.
- **Entrenar con los meses de 2025 ya pasados no ayudó.** En las regresiones útiles (`alfa` de 1 a 10.000) la opción 2 se equivoca entre 0,4 y 2,2 pasajeros por hora más de T2 a T4. En los árboles `poisson` y `residual` la diferencia queda entre -0,6 y +0,1, que es ruido, y `hgb_log_h63` empeora un poco hacia T4 (+0,8).
- **Las regresiones con `alfa` de 100.000 y 1.000.000 mejoran con la opción 2**, cada vez más hacia T4. No es por los datos nuevos, sino porque con más filas el mismo `alfa` castiga menos (sección 2).
- **Por qué los datos nuevos no ayudan.** El nivel de cada línea ya entra por los rezagos, así que el modelo se adapta solo si cambia la cantidad de pasajeros. Lo que aprende de train es la forma (cómo pesan la hora, el tipo de día, los feriados), y eso casi no cambió de 2024 a 2025. Que a Ridge hasta le vaya peor no lo pudimos explicar. Pasaba igual antes de la mediana limpia, así que no venía de ahí.

### 3.4 Guardar los resultados de validación

Van a la tabla `comparacion_validacion` con `etapa = 'validacion'`, una fila por opción y modelo. `seleccionado` marca el que eligió cada opción. Si se vuelve a correr, se reemplazan solo esas filas.

In [0]:
COLUMNAS = ["modelo", "filas", "mae", "mae_inf", "mae_sup", "rmse", "rmse_inf", "rmse_sup", "r2", "wape",
            "sesgo", "mejora_mae", "mejora_mae_inf", "mejora_mae_sup"]


def para_guardar(tabla, etapa, opcion, elegido):
    s = tabla[COLUMNAS].copy()
    s = s.astype({c: "float64" for c in COLUMNAS if c not in ("modelo", "filas")}).astype({"filas": "int64"})
    s.insert(0, "opcion", opcion)
    s.insert(0, "etapa", etapa)
    s["referencia"] = REFERENCIA
    s["seleccionado"] = (s["modelo"] == elegido).to_numpy()
    s["fecha_ejecucion"] = datetime.now().strftime("%Y-%m-%d %H:%M")
    return s


def guardar(partes, etapa):
    df = pd.concat(partes, ignore_index=True)
    escritor = spark.createDataFrame(df).write.format("delta").mode("overwrite").option("mergeSchema", "true")
    if spark.catalog.tableExists(config.TABLE_COMPARACION):
        escritor = escritor.option("replaceWhere", f"etapa = '{etapa}'")
    escritor.saveAsTable(config.TABLE_COMPARACION)
    print(f"Guardado en {config.TABLE_COMPARACION} (etapa '{etapa}'): {len(df)} filas")


guardar([para_guardar(r, "validacion", o, elegidos[o]) for o, r in resumen_val.items()], "validacion")

## 4. Antes de mirar test

Anotar acá, **antes** de prender el test, qué eligió cada opción y qué error prometió validación. Después de mirar test no se cambia nada: ni candidatos, ni ventanas, ni variables.

| | Opción 1 | Opción 2 |
|---|---|---|
| Modelo elegido | `hgb_poisson_h63` | `hgb_poisson_h63` |
| MAE en validación | 26,6 | 26,5 |
| Mejora sobre la mediana en validación | 33,1 % [25,8 % a 40,3 %] | 33,3 % [26,0 % a 40,5 %] |

Las dos eligen el mismo modelo, así que en test van a tener el mismo error. Lo que se puede comparar es cuál de las dos lo anticipó mejor (`optimismo`) y si el orden de los candidatos se mantiene (Spearman).

## 5. Test (2026): se corre UNA sola vez

1. Se entrenan todos los candidatos con **2024 y 2025** y se predice 2026. Como las dos opciones se reentrenan con los mismos datos, el modelo final de cada candidato es el mismo para las dos. Lo único que cambia entre opciones es **cuál** candidato eligieron.
2. Se calculan las reglas simples con las mismas filas.
3. Se compara lo que eligió cada opción.

Se predicen todos los candidatos (no solo los 2 elegidos) para poder ver si el orden de validación se mantiene en test. Eso es un **análisis posterior**: la elección ya quedó congelada en la sección 3 y no se cambia por lo que salga acá.

El test está apagado. Cuando estemos conformes con la sección 3, poner `EVALUAR_EN_TEST = True` y correr esta celda **una sola vez**.

In [0]:
EVALUAR_EN_TEST = False

if not EVALUAR_EN_TEST:
    print("Test apagado. Poner EVALUAR_EN_TEST = True cuando la sección 3 esté cerrada.")
else:
    test_raw = bajar(config.FECHA_INICIO_TEST, config.FECHA_FIN)
    meses = sorted(test_raw["fecha"].dt.strftime("%Y-%m").unique())
    print(f"Test: {len(test_raw):,} filas, {test_raw['fecha'].nunique()} días, meses {meses[0]} a {meses[-1]}")

    est_test, segundos = validacion.entrenar_y_medir(datos, test_raw, CANDIDATOS)
    resumen_test = validacion.resumen(est_test, REFERENCIA, n_rep=1000, seed=config.SEED)
    print("\nTodos los candidatos y las reglas en test:")
    print(validacion.para_mostrar(resumen_test).to_string(index=False))

    comparacion = validacion.tabla_comparacion(resumen_val, resumen_test, elegidos, REFERENCIA)
    print("\nComparación de las dos opciones:")
    print(comparacion.set_index("opcion").T.round(3).to_string())

    print("\n¿El orden de validación se mantiene en test?")
    for o in OPCIONES:
        rho, orden = validacion.concordancia(resumen_val[o], resumen_test, CANDIDATOS)
        print(f"\n{o}: correlación de Spearman entre puestos de val y de test = {rho:.2f}")
        print(orden.round(2).to_string(index=False))

    d = validacion.diferencia_entre_opciones(est_test, elegidos["opcion_2"], elegidos["opcion_1"],
                                             n_rep=1000, seed=config.SEED)
    print(f"\nMejora en test del modelo de la opción 2 sobre el de la opción 1: {100 * d['mejora']:.1f} % "
          f"[IC 95 %: {100 * d['inf']:.1f} % a {100 * d['sup']:.1f} %]. "
          "Positivo: la opción 2 se equivoca menos. Si el intervalo incluye el 0, no hay diferencia clara.")

    mostrar_test = list(dict.fromkeys([REFERENCIA, *elegidos.values()]))
    m = metricas.metricas(est_test[est_test["modelo"].isin(mostrar_test)], por=["tipo_dia"])
    tipos = [t for t in variables.TIPOS_DIA if t in set(m["tipo_dia"])]
    print("\nMAE por tipo de día en test:")
    print(m.pivot(index="modelo", columns="tipo_dia", values="mae").loc[mostrar_test, tipos].round(1).to_string())

    guardar([para_guardar(resumen_test, "test", o, elegidos[o]) for o in OPCIONES], "test")

**Cómo leer la comparación.**
- `mae_val` y `mae_test`: el error del modelo elegido en 2025 y en 2026.
- `test/val`: cuánto creció el error al pasar de validación a test.
- `optimismo`: ese crecimiento dividido el de la mediana de 4 semanas. La mediana no aprende nada, así que lo que cambia su error es solo que 2026 es distinto de 2025. Si `optimismo` es mayor a 1, la validación prometió menos error del que el modelo dio de verdad. La opción con `optimismo` más cerca de 1 es la que anticipa mejor el error real. Es una aproximación, porque supone que el cambio de 2025 a 2026 afecta igual al modelo y a la mediana.
- Spearman: cerca de 1, el orden de los candidatos en validación se mantiene en test. Cerca de 0 o negativo, la validación no sirvió para ordenarlos. Algunas regresiones son casi iguales entre sí (las de `alfa` más chico dan casi el mismo modelo) y pueden cambiar de puesto por centésimas. Si Spearman sale bajo, antes de sacar conclusiones hay que mirar la tabla de puestos y ver si los que se dieron vuelta son los parecidos o los distintos.
- Si las dos opciones eligen el mismo modelo, su error en test es el mismo. Igual se pueden comparar por `optimismo` y por Spearman.

Como 2026 llega hasta octubre, el test no tiene noviembre ni diciembre, y por eso conviene mirar la **mejora sobre la mediana** y el `optimismo`, que comparan contra la mediana en los mismos días, y no solo el MAE.

## Conclusiones

Completar con los resultados.

| Pregunta | Opción 1 (holdout de un año) | Opción 2 (validación cruzada) |
|---|---|---|
| Modelo elegido | | |
| MAE y mejora en validación (2025) | | |
| MAE y mejora en test (2026), con intervalo | | |
| `test/val` y `optimismo` | | |
| Spearman entre puestos de val y de test | | |

| Pregunta | Resultado |
|---|---|
| ¿Eligen el mismo modelo? Si no, ¿la diferencia en test es real (el intervalo excluye el 0)? | |
| ¿En qué trimestres de 2025 entrenar con datos más nuevos cambió el error? | |
| ¿Cuál anticipa mejor el error de 2026? | |
| ¿Algún modelo le gana a la mediana de 4 semanas en test? | |
| Entonces, ¿qué forma de dividir los datos recomendamos para este problema? | |

**Límites.**
- Hay un solo año de test, y llega hasta octubre.
- Algunas variables (los rezagos corregidos, el salto de clases y el paro) se agregaron después de mirar errores de 2025, así que la validación puede salir un poco optimista para las dos opciones.
- Para las últimas horas del día, el pronóstico del clima que usamos está justo en el límite de lo que se sabría la noche anterior (ver `clima.py`).
- Los candidatos son pocos: 10 regresiones Ridge que solo cambian en el castigo y en la combinación hora x tipo de día, y 3 árboles con un solo tamaño y 200 iteraciones fijas. Con más variantes de árboles la forma de validar podría pesar más en la elección.
- En la opción 2 el `alfa` de Ridge no se ajusta a la cantidad de filas, así que para los `alfa` grandes la comparación entre opciones no es pareja.
- La opción 2 tiene ventanas de 3 meses: cada una tiene pocos feriados, y algunos días especiales caen en una sola ventana.